## DOCUMENTATION BY: ONG LI XIANG DAAA/FT/2B/04

# Step by step guide on how I set up Part A

First, I set up my SCRUM Board.
The categories I have are:
- Backlog (for tasks that I have to do)
- In progress (tasks that I have started doing)
- Testing (double checking that it works before moving it to Done)
- Done (complete)


After cloning the project to my vsc, I create the .gitignore:

heres whats inside:
*.pyc
*.pem
*.pub
*.tar.gz
*.zip
*.sql
*.db
secrets.txt
./tmp
./build/*
.idea/*
.idea
env
venv
.vscode
__pycache__

## Now that the basic set up for vs code is done, I will load the GAN model from DELE CA1.

Looking at my saved weights, I saved the generator and discriminator separately as .h5 files

Since the assignment requires me to generate images I will only be using the generator model

I note that tensorflow expects the loaded model to be in a SavedModel format and not .h5 weights so we need to convert it first

The code for conversion is inside loaded_model.ipynb

## Now that we have the loaded model, we need to deploy the model with Docker.

These are the commands I ran in cmd 

docker pull python:3.8   
      
docker run -tid -v /var/run/docker.sock:/var/run/docker.sock --name CA2_Server python:3.8   

docker run --name ca2_gan_generator -p 8501:8501 -v "C:/Users/adven/OneDrive/Documents/school/DevOps/coding/ca2-daaa2b04-2317748-lixiangong/ai_model/gan_generator:/models/gan_generator/1" -e MODEL_NAME=gan_generator -t tensorflow/serving &
-e MODEL_NAME=gan_generator -t tensorflow/serving &


The first container (CA2_Server) provides a Python environment for any additional processing or management tasks.
    
The second container (ca2_gan_generator) is dedicated to serving the TensorFlow model, making it accessible for predictions via a REST API.

## We will now perform a health check to make sure the model is deployed properly by going to this url:

http://localhost:8501/v1/models/generator

This is what we get from the site:

{
 "model_version_status": [
  {
   "version": "1",
   "state": "AVAILABLE",
   "status": {
    "error_code": "OK",
    "error_message": ""
   }
  }
 ]
}

# We verified that its deployed properly but is it working as a GAN?

## To do that we will do a basic unit test to ensure that it works properly as a GAN. I set 3 different types of tests:

## Test Output Validity
### Purpose: This test checks the validity of the output generated by the GAN model.
How?
- It generates random latent vectors as input to the GAN.
- It sends these vectors to the model and retrieves the predictions.
- It asserts that the output is a list containing two images.
- It verifies that each image has the expected dimensions (28x28 pixels with 1 channel, typical for grayscale images like MNIST).



## Test Pixel Range
### Purpose: This test ensures that the pixel values of the generated images fall within the expected range.
How?
- It generates a single set of latent vectors and retrieves the corresponding predictions.
- It flattens the pixel values of the first generated image.
- It asserts that all pixel values are within the range of [-1, 1], which is typical for GAN outputs that use a tanh activation function.



## Test Invalid Input Handling
### Purpose: This test checks how the model handles invalid input data.
How?
- Test 1: It sends a malformed input (a string instead of a number) to the model and asserts that the response status code is 400 (Bad Request), indicating that the model correctly rejected the invalid input.
- Test 2: It sends a latent vector with an incorrect dimension and checks that the model also responds with a 400 status code, confirming that it handles dimensionality errors appropriately.

### To run these tests, go to the directory with test_docker.py and run
pytest test_docker.py -v

Here is what a successful output looks like:

PS C:\Users\adven\OneDrive\Documents\school\DevOps\coding\ca2-daaa2b04-2317748-lixiangong\tests> pytest test_docker.py -v
========================================================================================================= test session starts =========================================================================================================
platform win32 -- Python 3.10.11, pytest-8.3.4, pluggy-1.5.0 -- C:\Users\adven\AppData\Local\Programs\Python\Python310\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\adven\OneDrive\Documents\school\DevOps\coding\ca2-daaa2b04-2317748-lixiangong\tests
collected 3 items

test_docker.py::test_output_validity PASSED                                                                                                                                                                                      [ 33%]
test_docker.py::test_pixel_range PASSED                                                                                                                                                                                          [ 66%]
test_docker.py::test_invalid_input_handling PASSED                                                                                                                                                                               [100%]

========================================================================================================== 3 passed in 1.02s ==========================================================================================================
PS C:\Users\adven\OneDrive\Documents\school\DevOps\coding\ca2-daaa2b04-2317748-lixiangong\tests> 

### After basic unit testing, a Postman test

type: POST

url: https://ca2-daaa2b04-2317748-lixiangong.onrender.com/v1/models/generator:predict

json body:

{
    "signature_name": "serving_default",
    "instances": [
        [0.1, -0.2, 0.3, -0.4, 0.5, -0.6, 0.7, -0.8, 0.9, -0.1,
         0.2, -0.3, 0.4, -0.5, 0.6, -0.7, 0.8, -0.9, 0.0, 0.1,
         0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, -0.1, -0.2,
         -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, 0.0, 0.1,
         0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, -0.1, -0.2,
         -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, 0.0, 0.1,
         0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, -0.1, -0.2,
         -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, 0.0, 0.1,
         0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, -0.1, -0.2,
         -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, 0.0, 0.1,
         0.2, 0.3, 0.4, 0.5]  // Total of 96 values
    ]
}

and I got a 200 OK request showing that the predict endpoint works as expected

### Now we can prepare for deployment on Render

We start by creating a Dockerfile and then after that, going on Render and adding a new Web Service linking it to our project on GitLab and then deploy it

We can check that its deployed successfully by going on the Render dashboard and checking that there is a green Deployed button

### Now that its deployed successfully the last thing we need to do is update the test_docker predict url to the new updated render one to test if it still works.

old url: http://localhost:8501/v1/models/generator:predict
new url: https://ca2-daaa2b04-2317748-lixiangong.onrender.com/v1/models/generator:predict

running the new url test_docker.py

output:

PS C:\Users\adven\OneDrive\Documents\school\DevOps\coding\ca2-daaa2b04-2317748-lixiangong\tests> pytest test_docker.py -v
========================================================================================================= test session starts =========================================================================================================
platform win32 -- Python 3.10.11, pytest-8.3.4, pluggy-1.5.0 -- C:\Users\adven\AppData\Local\Programs\Python\Python310\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\adven\OneDrive\Documents\school\DevOps\coding\ca2-daaa2b04-2317748-lixiangong\tests
collected 3 items

test_docker.py::test_output_validity PASSED                                                                                                                                                                                      [ 33%]
test_docker.py::test_pixel_range PASSED                                                                                                                                                                                          [ 66%]
test_docker.py::test_invalid_input_handling PASSED                                                                                                                                                                               [100%]

========================================================================================================== 3 passed in 1.02s ==========================================================================================================
PS C:\Users\adven\OneDrive\Documents\school\DevOps\coding\ca2-daaa2b04-2317748-lixiangong\tests> 


it works!